# 📊 Notebook 4 — Avaliação (Etapa 11)

Avalia o sistema RAG em duas frentes:

1. **Mini-benchmark interno** com 25 perguntas simuladas cobrindo os 5 tipos do PDF:
   - Vigência | Conteúdo Normativo | Motivação | Dados Tarifários | Relacionamento

2. **Métricas RAGAS** automáticas:
   - `faithfulness` — a resposta é sustentada pelo contexto?
   - `answer_relevancy` — a resposta endereça a pergunta?
   - `context_precision` — o contexto recuperado é útil?

3. **Função pronta para o benchmark oficial** quando chegar.

**Pré-requisito:** Notebook 3 executado (Qdrant + chunks + LLMs configurados)

In [ ]:
# ============================================================
# CÉLULA 1 — Instalação
# ============================================================
# IMPORTANTE: rodar esta célula DEPOIS de já ter executado o notebook 3
# (que tem o pipeline e os modelos carregados).
%%capture
!pip install ragas datasets langchain-groq langchain-huggingface

In [ ]:
# ============================================================
# CÉLULA 2 — Verifica se o pipeline do Notebook 3 está carregado
# ============================================================
# Este notebook ASSUME que você executou o notebook 3 nesta mesma sessão Colab.
# Se reiniciou, rode primeiro as células 1-11 do notebook 3.

try:
    _ = ask  # função do notebook 3
    _ = hybrid_retrieve
    _ = GROQ_API_KEY
    print('✅ Pipeline do Notebook 3 detectado')
except NameError as e:
    raise RuntimeError(
        'Execute primeiro o notebook 03_rag_pipeline.ipynb (células 1-11) '
        'na MESMA sessão Colab antes de rodar este notebook.'
    ) from e

In [ ]:
# ============================================================
# CÉLULA 3 — Mini-benchmark interno (25 perguntas)
# ============================================================
# Cobre os 5 tipos identificados no PDF:
#   1. Vigência (situação atual de uma resolução)
#   2. Conteúdo normativo (o que a norma estabelece)
#   3. Motivação (por que decidiu)
#   4. Dados tarifários (valores específicos)
#   5. Relacionamento (resoluções que se conectam)

MINI_BENCHMARK = [
    # --- 1. VIGÊNCIA ---
    {'tipo': 'vigencia', 'pergunta': 'A Resolução Normativa nº 414/2010 ainda está em vigor?'},
    {'tipo': 'vigencia', 'pergunta': 'Quais resoluções normativas da ANEEL foram revogadas em 2022?'},
    {'tipo': 'vigencia', 'pergunta': 'A REN 482/2012 sobre micro e minigeração está vigente?'},
    {'tipo': 'vigencia', 'pergunta': 'Quando entrou em vigor o novo regulamento de comercialização de energia?'},
    {'tipo': 'vigencia', 'pergunta': 'A norma sobre revisão tarifária extraordinária ainda se aplica?'},

    # --- 2. CONTEÚDO NORMATIVO ---
    {'tipo': 'conteudo', 'pergunta': 'O que estabelece a Resolução Normativa sobre micro e minigeração distribuída?'},
    {'tipo': 'conteudo', 'pergunta': 'Quais são as regras para suspensão do fornecimento de energia elétrica por inadimplência?'},
    {'tipo': 'conteudo', 'pergunta': 'Como deve ser feito o ressarcimento por danos elétricos a consumidores?'},
    {'tipo': 'conteudo', 'pergunta': 'Quais são os procedimentos para conexão de novas unidades consumidoras à rede?'},
    {'tipo': 'conteudo', 'pergunta': 'Quais são as obrigações das distribuidoras quanto à qualidade do serviço?'},

    # --- 3. MOTIVAÇÃO ---
    {'tipo': 'motivacao', 'pergunta': 'Por que a ANEEL revisou as regras de comercialização em 2021?'},
    {'tipo': 'motivacao', 'pergunta': 'Qual a justificativa técnica para alteração da bandeira tarifária?'},
    {'tipo': 'motivacao', 'pergunta': 'Por que foi criado o Mecanismo de Realocação de Energia (MRE)?'},
    {'tipo': 'motivacao', 'pergunta': 'Qual o fundamento para os reajustes tarifários extraordinários durante crises hídricas?'},
    {'tipo': 'motivacao', 'pergunta': 'Por que a ANEEL flexibilizou regras para geração distribuída?'},

    # --- 4. DADOS TARIFÁRIOS ---
    {'tipo': 'tarifa', 'pergunta': 'Qual foi o valor da TUSD aplicada às distribuidoras em 2022?'},
    {'tipo': 'tarifa', 'pergunta': 'Qual o reajuste tarifário aprovado para a CEMIG em 2016?'},
    {'tipo': 'tarifa', 'pergunta': 'Quais foram os índices de qualidade DEC e FEC aprovados em 2021?'},
    {'tipo': 'tarifa', 'pergunta': 'Qual o valor do encargo de Conta de Desenvolvimento Energético (CDE) em 2022?'},
    {'tipo': 'tarifa', 'pergunta': 'Qual percentual de reajuste foi aplicado na bandeira vermelha em 2021?'},

    # --- 5. RELACIONAMENTO ENTRE ATOS ---
    {'tipo': 'relacao', 'pergunta': 'Quais resoluções normativas alteraram a REN 414/2010 ao longo do tempo?'},
    {'tipo': 'relacao', 'pergunta': 'A REN 482/2012 foi modificada por quais atos normativos posteriores?'},
    {'tipo': 'relacao', 'pergunta': 'Quais resoluções a REN 1000/2021 revogou expressamente?'},
    {'tipo': 'relacao', 'pergunta': 'Qual a relação entre as regras de PRORET e as resoluções normativas de tarifa?'},
    {'tipo': 'relacao', 'pergunta': 'Quais notas técnicas fundamentaram a Resolução de Bandeiras Tarifárias?'},
]

print(f'✅ {len(MINI_BENCHMARK)} perguntas no mini-benchmark interno')
print(f'   Distribuição por tipo:')
from collections import Counter
for tipo, count in Counter([q['tipo'] for q in MINI_BENCHMARK]).items():
    print(f'     {tipo}: {count}')

In [ ]:
# ============================================================
# CÉLULA 4 — Roda o mini-benchmark com ambos os LLMs
# ============================================================
import json
import time
from tqdm.notebook import tqdm

def run_benchmark(
    perguntas: list,
    llm: str = 'groq',
    delay: float = 1.5,
    output_file: str = None,
):
    """
    Roda lista de perguntas, retorna lista de resultados.

    Args:
        perguntas: list[dict] com 'pergunta' (e opcionalmente 'tipo', 'resposta_ref')
                   ou list[str] de perguntas
        llm: 'maritaca' ou 'groq'
        delay: segundos entre requisições (rate limit)
        output_file: path opcional para salvar JSON
    """
    results = []
    for i, item in enumerate(tqdm(perguntas, desc=f'Benchmark ({llm})')):
        # Aceita string ou dict
        if isinstance(item, dict):
            q = item.get('pergunta') or item.get('question')
            tipo_pergunta = item.get('tipo', '')
            resposta_ref = item.get('resposta_ref') or item.get('answer')
        else:
            q, tipo_pergunta, resposta_ref = str(item), '', None

        try:
            answer, retrieved = ask(q, llm=llm, verbose=False, return_chunks=True)
            sources = [
                {
                    'ato_id':         r[2]['ato_id'],
                    'tipo_documento': r[2]['tipo_documento'],
                    'titulo':         r[2]['titulo'][:100],
                    'situacao':       r[2]['situacao'],
                    'publicacao':     r[2]['publicacao'],
                    'score':          float(r[1]),
                }
                for r in retrieved
            ]
            contexts = [(r[2].get('texto_pai') or r[2]['texto']) for r in retrieved]

            entry = {
                'pergunta':      q,
                'tipo_pergunta': tipo_pergunta,
                'resposta':      answer,
                'fontes':        sources,
                'contextos':     contexts,
                'llm':           llm,
            }
            if resposta_ref:
                entry['resposta_ref'] = resposta_ref
        except Exception as e:
            entry = {'pergunta': q, 'erro': str(e), 'llm': llm}

        results.append(entry)
        if delay > 0 and i < len(perguntas) - 1:
            time.sleep(delay)

    if output_file:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f'💾 Salvo: {output_file}')

    return results


# ───── Roda o mini-benchmark ─────
print('🚀 Rodando mini-benchmark com Groq...')
results_groq = run_benchmark(
    MINI_BENCHMARK, llm='groq',
    output_file='/content/results_groq.json',
)

if maritaca_client:
    print('\n🚀 Rodando mini-benchmark com Maritaca...')
    results_maritaca = run_benchmark(
        MINI_BENCHMARK, llm='maritaca',
        output_file='/content/results_maritaca.json',
    )
else:
    print('\n⚠️  Maritaca não configurada — pulando.')
    results_maritaca = []

print(f'\n✅ Concluído! Groq: {len(results_groq)} | Maritaca: {len(results_maritaca)}')

In [ ]:
# ============================================================
# CÉLULA 5 — Setup do RAGAS
# ============================================================
# RAGAS usa um LLM para avaliar — usaremos Groq (gratuito) como avaliador.
# Embeddings: e5-base (mais rápido que large) só para os métricas RAGAS.

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=GROQ_API_KEY,
    temperature=0,
))

ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(
    model_name='intfloat/multilingual-e5-base',
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'},
))

print('✅ RAGAS configurado (LLM avaliador: Groq)')

In [ ]:
# ============================================================
# CÉLULA 6 — Avaliação RAGAS
# ============================================================
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)

def to_ragas_dataset(results: list) -> Dataset:
    """Converte os resultados do benchmark para o formato esperado pelo RAGAS."""
    data = {
        'user_input':        [r['pergunta']  for r in results if 'erro' not in r],
        'response':          [r['resposta']  for r in results if 'erro' not in r],
        'retrieved_contexts': [r['contextos'] for r in results if 'erro' not in r],
    }
    return Dataset.from_dict(data)


def avaliar_com_ragas(results: list, label: str = ''):
    """Calcula métricas RAGAS sobre os resultados de um LLM."""
    if not results:
        print(f'⚠️  Sem resultados para {label}')
        return None

    print(f'\n📊 Avaliando RAGAS: {label} ({len(results)} perguntas)')
    ragas_ds = to_ragas_dataset(results)

    metrics = [
        Faithfulness(llm=ragas_llm),
        ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
        LLMContextPrecisionWithoutReference(llm=ragas_llm),
    ]

    scores = evaluate(
        dataset=ragas_ds,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_emb,
        raise_exceptions=False,
    )
    return scores


# ───── Avalia ambos ─────
print('⏱️  RAGAS pode demorar (rate limit Groq) — ~2min por LLM')
scores_groq = avaliar_com_ragas(results_groq, 'GROQ Llama 3.3 70B')
scores_maritaca = avaliar_com_ragas(results_maritaca, 'MARITACA Sabiá-3') if results_maritaca else None

In [ ]:
# ============================================================
# CÉLULA 7 — Comparação Maritaca vs Groq
# ============================================================
import pandas as pd

def to_summary(scores, label):
    if scores is None:
        return None
    df = scores.to_pandas()
    summary = {
        'LLM':              label,
        'faithfulness':     df['faithfulness'].mean(),
        'answer_relevancy': df['answer_relevancy'].mean(),
        'context_precision': df['llm_context_precision_without_reference'].mean(),
    }
    summary['média_geral'] = np.mean([
        summary['faithfulness'],
        summary['answer_relevancy'],
        summary['context_precision'],
    ])
    return summary

summaries = []
if scores_groq:     summaries.append(to_summary(scores_groq, 'Groq Llama 3.3'))
if scores_maritaca: summaries.append(to_summary(scores_maritaca, 'Maritaca Sabiá-3'))

if summaries:
    df_comp = pd.DataFrame(summaries).round(3)
    print('🏆 RESULTADO COMPARATIVO RAGAS')
    print('=' * 80)
    print(df_comp.to_string(index=False))
    print('\n📌 Métricas:')
    print('   faithfulness     = a resposta é fiel ao contexto recuperado?')
    print('   answer_relevancy = a resposta endereça a pergunta?')
    print('   context_precision = o contexto recuperado é útil?')

    melhor = df_comp.iloc[df_comp['média_geral'].idxmax()]
    print(f'\n🥇 Vencedor: {melhor["LLM"]} (média {melhor["média_geral"]:.3f})')

In [ ]:
# ============================================================
# CÉLULA 8 — Análise qualitativa: amostra de respostas
# ============================================================
import random

random.seed(42)
amostra_idx = random.sample(range(len(results_groq)), min(5, len(results_groq)))

for idx in amostra_idx:
    r_g = results_groq[idx]
    print(f"\n{'='*70}")
    print(f"❓ [{r_g.get('tipo_pergunta','').upper()}] {r_g['pergunta']}")
    print('=' * 70)

    if 'resposta' in r_g:
        print(f'\n🦙 GROQ:\n{r_g["resposta"]}\n')
        print('Fontes:')
        for s in r_g['fontes'][:3]:
            print(f"  - [{s['tipo_documento']}] {s['ato_id']} | {s['publicacao']} | {s['situacao']}")

    if results_maritaca and idx < len(results_maritaca):
        r_m = results_maritaca[idx]
        if 'resposta' in r_m:
            print(f'\n🦫 MARITACA:\n{r_m["resposta"]}\n')

In [ ]:
# ============================================================
# CÉLULA 9 — 🎯 Função para o BENCHMARK OFICIAL (do desafio)
# ============================================================
# Quando receberem as perguntas oficiais, basta colar e rodar.

def rodar_benchmark_oficial(
    perguntas: list,
    llm: str = 'maritaca',
    salvar_em: str = '/content/drive/MyDrive/aneel_rag/respostas_benchmark.json',
):
    """
    Roda o benchmark oficial e salva no Drive.

    Aceita:
      - lista de strings: ['pergunta 1', 'pergunta 2', ...]
      - lista de dicts:   [{'pergunta': '...', 'tipo': '...'}, ...]
      - JSON file path:   'perguntas.json' (será carregado automaticamente)
    """
    if isinstance(perguntas, str) and os.path.exists(perguntas):
        with open(perguntas, encoding='utf-8') as f:
            perguntas = json.load(f)

    print(f'🎯 Rodando benchmark oficial: {len(perguntas)} perguntas com {llm.upper()}')
    resultados = run_benchmark(perguntas, llm=llm, output_file=salvar_em)
    print(f'\n✅ Respostas salvas em: {salvar_em}')
    return resultados

# ─── Quando receber o benchmark, descomente e rode ───
# benchmark_oficial = [
#     'Pergunta 1 do desafio...',
#     'Pergunta 2 do desafio...',
# ]
# resultados_oficiais = rodar_benchmark_oficial(benchmark_oficial, llm='maritaca')

print('ℹ️  Função rodar_benchmark_oficial() pronta para uso')

In [ ]:
# ============================================================
# CÉLULA 10 — Salva relatório de avaliação
# ============================================================
DRIVE_DIR = '/content/drive/MyDrive/aneel_rag'

relatorio = {
    'mini_benchmark_groq':     results_groq,
    'mini_benchmark_maritaca': results_maritaca,
    'ragas_summary':           summaries if summaries else None,
    'configuracao': {
        'embedding_model': EMBEDDING_MODEL,
        'reranker_model':  RERANKER_MODEL,
        'final_k':         FINAL_K,
        'alpha_rrf':       ALPHA_RRF,
        'tipo_boost':      TIPO_BOOST,
    }
}

out_path = f'{DRIVE_DIR}/relatorio_avaliacao.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2, default=str)
print(f'💾 Relatório salvo: {out_path}')

print('\n🎉 PIPELINE COMPLETO!')
print('   ✅ Chunking hierárquico')
print('   ✅ Embeddings e5-large-instruct')
print('   ✅ Indexação Qdrant')
print('   ✅ Retrieval híbrido + cross-encoder + expansão por ato_id')
print('   ✅ LLMs (Maritaca + Groq)')
print('   ✅ Avaliação RAGAS')